In [1]:
import pandas as pd
import numpy as np


In [2]:
day="5th August"
data=pd.read_csv('aug5.csv')
data.head()

,Timestamp,Name,Roll No,Distance
0,8/5/2026 16:51:24,Test,Test,294 meters away
1,8/5/2026 17:35:23,MUKUL Dahiya,B26544,8 meters away
2,8/5/2026 17:35:29,Anshul Kalia,IM26031,11 meters away
3,8/5/2026 17:35:34,Lakshay Garg,B26137,12 meters away
4,8/5/2026 17:35:39,Rehan hasan,B26404,11 meters away


In [3]:
output=pd.read_csv("complete_attendance_v1.csv")
output.head()

,Roll Number,3rd August,4th August,5th August,6th August,7th August,8th August,9th August
0,B26001,1,1,0,0,0,0,0
1,B26002,1,1,0,0,0,0,0
2,B26003,1,1,0,0,0,0,0
3,B26004,1,1,0,0,0,0,0
4,B26005,1,1,0,0,0,0,0


In [5]:
roll=data['Roll No'].tolist()
for i in range(len(roll)):
    roll[i]=roll[i].strip()
    roll[i]=roll[i].upper()
print(roll)
    
    

['TEST', 'B26544', 'IM26031', 'B26137', 'B26404', 'B26343', 'B26417', 'B26177', 'B26555', 'B26547', 'B26619', 'B26216', 'B26631', 'B26248', 'B26117', 'B26590', 'B26107', 'B26458', 'B26251', 'B26103', 'B26116', 'B26126', 'B26243', 'B26588', 'B26315', 'B26591', 'B26358', 'B26549', 'B26560', 'B26105', 'B26140', 'B26492', 'B26250', 'B26325', 'B26155', 'B26281', 'B26542', 'B26242', 'B26237', 'B26556', 'B26160', 'IM26014', 'B26609', 'B26121', 'B26576', 'B26427', 'B26205', 'B26199', 'B26399', 'B26214', 'B26215', 'B26189', 'IM26012', 'B26582', 'B26229', 'B26457', 'B26071', 'IM26029', 'B26269', 'B26040', 'B26502', 'B26112', 'B26234', 'B26136', 'B26365', 'B26026', 'B26442', 'B26423', 'B26561', 'B26184', 'B26361', 'B26527', 'B26062', 'B26095', 'B26151', 'B26196', 'B26055', 'B26246', 'B26285', 'B26070', 'B26308', 'B26311', 'B26198', 'B26047', 'B26545', 'B26108', 'B26211', 'B26128', 'B26178', 'B26195', 'B26368', 'B26114', 'B26003', 'B26558', 'IM26079', 'B26159', 'B26391', 'B26144', 'B26271', 'B2642

In [6]:
def generate_attendance():
    for i in roll:
        output.loc[output['Roll Number']==i,day]=1
    print(output.head(100))
    output.to_csv("complete_attendance_v1.csv",index=False)   
        

In [7]:
generate_attendance()

   Roll Number 3rd August  4th August  5th August  6th August  7th August  \
0       B26001          1           1           1           0           0   
1       B26002          1           1           1           0           0   
2       B26003          1           1           1           0           0   
3       B26004          1           1           1           0           0   
4       B26005          1           1           1           0           0   
..         ...        ...         ...         ...         ...         ...   
95      B26096          1           1           1           0           0   
96      B26097          1           1           1           0           0   
97      B26098          0           0           0           0           0   
98      B26099          1           1           1           0           0   
99      B26100          1           1           1           0           0   

    8th August  9th August  
0            0           0  
1            0   

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

def analyze_and_plot_attendance(file_path, day='3rd August'):
    # 1. Load Data
    df = pd.read_csv(file_path)

    # 2. Categorize Program based on Roll Number prefix
    def get_program(roll):
        roll_str = str(roll).strip()
        if roll_str.startswith('B26'):
            return 'BTech'
        elif roll_str.startswith('IM26'):
            return 'IMBA'
        else:
            return 'Other'

    df['Program'] = df['Roll Number'].apply(get_program)
    date_cols = [c for c in df.columns if c not in ['Roll Number', 'Program']]

    # Check if specified day exists in the dataset
    if day not in date_cols:
        print(f"Error: '{day}' was not found in the dataset dates.")
        print(f"Available dates: {date_cols}")
        return

    # 3. Reshape Data (Long format)
    df_long = df.melt(
        id_vars=['Roll Number', 'Program'], 
        value_vars=date_cols, 
        var_name='Date', 
        value_name='Status'
    )
    
    # Normalize status labels
    df_long['Status'] = df_long['Status'].astype(str).str.strip().str.capitalize()
    status_map = {'1': 'Present', '0': 'Absent', 'Proxy': 'Proxy'}
    df_long['Status'] = df_long['Status'].map(status_map).fillna(df_long['Status'])

    # Filter data for the requested day
    selected_day_df = df_long[df_long['Date'] == day]

    # --- PRINT SUMMARY STATISTICS ---
    print("=" * 60)
    print(f" SUMMARY STATISTICS FOR {day.upper()}")
    print("=" * 60)
    
    print("\n1. TOTAL STUDENT COUNT BY PROGRAM:")
    print(df['Program'].value_counts().to_string())

    print(f"\n2. ATTENDANCE SUMMARY FOR {day}:")
    day_summary = selected_day_df.groupby(['Program', 'Status']).size().unstack(fill_value=0)
    print(day_summary.to_string())

    print(f"\n3. PROXY INCIDENTS DETECTED ON {day}:")
    proxies = selected_day_df[selected_day_df['Status'] == 'Proxy']
    if not proxies.empty:
        print(proxies[['Roll Number', 'Program', 'Date']].to_string(index=False))
    else:
        print(f"No proxy attendance found on {day}.")
    print("=" * 60)

    # File naming helper (converts "3rd August" -> "3rd_august")
    safe_day_str = day.lower().replace(' ', '_')

    # --- GENERATE & SAVE GRAPHS ---

    # Graph 1: Program Distribution (Pie Chart)
    plt.figure(figsize=(6, 6))
    program_counts = df['Program'].value_counts()
    plt.pie(
        program_counts, 
        labels=program_counts.index, 
        autopct='%1.1f%%', 
        colors=['#4C72B0', '#DD8452'], 
        startangle=140, 
        explode=(0.05, 0)
    )
    plt.title('Student Distribution by Program', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('program_distribution.png', dpi=300)
    plt.close()
    print(" Saved: program_distribution.png")

    # Graph 2: Attendance Comparison on Specified Day (Bar Chart)
    plt.figure(figsize=(8, 5))
    ax = sns.countplot(
        data=selected_day_df, 
        x='Program', 
        hue='Status', 
        palette={'Present': '#55A868', 'Absent': '#C44E52', 'Proxy': '#8172B3'}
    )
    plt.title(f'Attendance Status Comparison ({day})', fontsize=14, fontweight='bold')
    plt.xlabel('Program', fontsize=12)
    plt.ylabel('Number of Students', fontsize=12)
    
    # Add data values on top of bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')
    
    plt.tight_layout()
    day_bar_filename = f'{safe_day_str}_attendance.png'
    plt.savefig(day_bar_filename, dpi=300)
    plt.close()
    print(f" Saved: {day_bar_filename}")

    # Graph 3: Date-wise Attendance Trend (Line Chart across all days)
    plt.figure(figsize=(10, 5))
    trend_df = df_long.groupby(['Date', 'Status']).size().reset_index(name='Count')
    sns.lineplot(
        data=trend_df, 
        x='Date', 
        y='Count', 
        hue='Status', 
        marker='o', 
        linewidth=2.5, 
        palette={'Present': '#55A868', 'Absent': '#C44E52', 'Proxy': '#8172B3'}
    )
    plt.title('Overall Attendance Trend Across All Dates', fontsize=14, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Total Count', fontsize=12)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig('attendance_trend.png', dpi=300)
    plt.close()
    print(" Saved: attendance_trend.png")

# Run analysis for a specific day
if __name__ == "__main__":
    # Change 'day' variable to any date present in your dataset 
    # Available options: '3rd August', '4th August', '5th August', '6th August', '7th August', '8th August', '9th August'
    target_day = "5th August"
    
    analyze_and_plot_attendance('complete_attendance_v1.csv', day=target_day)

 SUMMARY STATISTICS FOR 5TH AUGUST

1. TOTAL STUDENT COUNT BY PROGRAM:
Program
BTech    640
IMBA      80

2. ATTENDANCE SUMMARY FOR 5th August:
Status   Absent  Present
Program                 
BTech        66      574
IMBA          7       73

3. PROXY INCIDENTS DETECTED ON 5th August:
No proxy attendance found on 5th August.
 Saved: program_distribution.png
 Saved: 5th_august_attendance.png
 Saved: attendance_trend.png
